# 02 — Attribute Label Generation
## Multi-Attribute Scene Classification on nuScenes Front-Camera Images

**This notebook:** Generates the four scene-attribute labels for every CAM_FRONT keyframe. These labels are the targets for the four parallel classification tasks.

### The four attributes

| # | Attribute | Classes | Source |
|---|---|---|---|
| 1 | `time_of_day` | day / night | Scene description text + brightness statistics |
| 2 | `weather` | clear / rain | Scene description text |
| 3 | `vehicle_density` | low / medium / high | Tertile bins on forward-cone vehicle count |
| 4 | `vru_present` | absent / present | Forward-cone VRU count > 0 |

### Outputs

| File | Contents |
|---|---|
| `data/labels/attribute_labels.csv` | One row per keyframe with all four labels |
| `data/labels/attribute_thresholds.json` | Vehicle-density bin edges + any other thresholds |
| `results/figures/labels/*.png` | Class-distribution plots per attribute |


## 0. Setup

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

DATASET_VERSION = 'v1.0-mini'   # change to 'v1.0-trainval' when scaling up
META_DIR  = Path('data/metadata')
LABEL_DIR = Path('data/labels')
FIG_DIR   = Path('results/figures/labels')
for p in [LABEL_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f'DATASET_VERSION = {DATASET_VERSION}')

In [ ]:
# Load enriched metadata from notebook 01
df = pd.read_csv(META_DIR / 'sample_metadata_enriched.csv')
print(f'Loaded {len(df)} keyframes from notebook 01')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 1. Time-of-Day Label

**Rule:** Match the scene-description text. Default = 'day'. If the description contains a night keyword → 'night'.

We rely on the scene-description text rather than brightness alone because (a) it's explicit ground truth from the dataset annotators, and (b) brightness can be misleading (e.g., a dark tunnel during the day, or a brightly-lit night scene).

In [ ]:
NIGHT_KEYWORDS = ['night']
def classify_time_of_day(desc: str) -> str:
    d = (desc or '').lower()
    return 'night' if any(k in d for k in NIGHT_KEYWORDS) else 'day'

df['time_of_day'] = df['scene_description'].apply(classify_time_of_day)
print('Time-of-day distribution:')
print(df['time_of_day'].value_counts())

## 2. Weather Label

**Rule:** Match the scene-description text. Default = 'clear'. If the description contains a rain keyword → 'rain'.

We deliberately collapse 'rain', 'wet', 'after rain' into a single 'rain' class to avoid label fragmentation on a small dataset.

In [ ]:
RAIN_KEYWORDS = ['rain', 'wet']
def classify_weather(desc: str) -> str:
    d = (desc or '').lower()
    return 'rain' if any(k in d for k in RAIN_KEYWORDS) else 'clear'

df['weather'] = df['scene_description'].apply(classify_weather)
print('Weather distribution:')
print(df['weather'].value_counts())

## 3. Vehicle-Density Label

**Rule:** Tertile binning of `veh_count_fwd` (forward-cone vehicle count).
- low = bottom third
- medium = middle third
- high = top third

Tertile binning is preferable to fixed thresholds because the count distribution depends on dataset (mini vs trainval). Tertiles automatically rescale and yield ~33%/33%/33% class balance — easier to learn than a heavily skewed split.

**Caveat:** When two tertile boundaries coincide (e.g., many keyframes have count=0), we shift bins so the rare side keeps at least one sample. Edge case handled below.

In [ ]:
def tertile_bin_label(s: pd.Series, labels=('low', 'medium', 'high')) -> tuple[pd.Series, list]:
    """Return tertile-binned labels and the cut points used."""
    # Use rank-based tertiles to handle ties in integer counts
    ranks = s.rank(method='first', ascending=True)
    tertiles = pd.qcut(ranks, q=3, labels=labels)
    # Recover edges (in original count units) from the data
    n = len(s)
    sorted_s = s.sort_values().reset_index(drop=True)
    edge_low = float(sorted_s.iloc[max(int(n/3) - 1, 0)])
    edge_high = float(sorted_s.iloc[max(int(2*n/3) - 1, 0)])
    return tertiles.astype(str), [edge_low, edge_high]

df['vehicle_density'], veh_edges = tertile_bin_label(df['veh_count_fwd'])

print(f'Tertile edges (vehicle_count_fwd): low ≤ {veh_edges[0]} | medium ≤ {veh_edges[1]} | high > {veh_edges[1]}')
print(f'\nVehicle-density distribution:')
print(df['vehicle_density'].value_counts())

## 4. VRU-Presence Label

**Rule:** binary `vru_present` = 1 if `vru_count_fwd >= 1` else 0.

We use a single threshold (≥1) rather than a count-based scheme because (a) most images have 0 or 1 VRU, making count-based bins thin, and (b) for safety-critical AV reasoning, the binary "is there a VRU at all?" signal is the operationally meaningful one.

In [ ]:
df['vru_present'] = (df['vru_count_fwd'] >= 1).astype(int).map({0: 'absent', 1: 'present'})
print('VRU-presence distribution:')
print(df['vru_present'].value_counts())

## 5. Assemble Final Labels Table

In [ ]:
label_cols = ['sample_token', 'scene_token', 'scene_name', 'image_path',
              'time_of_day', 'weather', 'vehicle_density', 'vru_present',
              'veh_count_fwd', 'vru_count_fwd', 'scene_description']
df_labels = df[label_cols].copy()

print(f'Final labels table: {len(df_labels)} rows × {len(df_labels.columns)} columns')
df_labels.head()

In [ ]:
df_labels.to_csv(LABEL_DIR / 'attribute_labels.csv', index=False)

# Save threshold metadata for reproducibility
thresholds = {
    'dataset_version': DATASET_VERSION,
    'time_of_day': {
        'rule': 'scene_description contains a night keyword',
        'night_keywords': NIGHT_KEYWORDS,
        'default': 'day',
    },
    'weather': {
        'rule': 'scene_description contains a rain keyword',
        'rain_keywords': RAIN_KEYWORDS,
        'default': 'clear',
    },
    'vehicle_density': {
        'rule': 'tertile binning of forward-cone vehicle count',
        'edges': {'low_max': veh_edges[0], 'medium_max': veh_edges[1]},
    },
    'vru_present': {
        'rule': 'forward-cone VRU count >= 1',
        'threshold': 1,
    },
}
with open(LABEL_DIR / 'attribute_thresholds.json', 'w') as f:
    json.dump(thresholds, f, indent=2)

print(f'Saved labels      → {LABEL_DIR / "attribute_labels.csv"}')
print(f'Saved thresholds  → {LABEL_DIR / "attribute_thresholds.json"}')

## 6. Class-Distribution Plots

In [ ]:
attributes = ['time_of_day', 'weather', 'vehicle_density', 'vru_present']
class_orders = {
    'time_of_day':     ['day', 'night'],
    'weather':         ['clear', 'rain'],
    'vehicle_density': ['low', 'medium', 'high'],
    'vru_present':     ['absent', 'present'],
}
colors = ['#3498db', '#9b59b6', '#27ae60', '#e67e22']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, attr, color in zip(axes, attributes, colors):
    counts = df_labels[attr].value_counts().reindex(class_orders[attr]).fillna(0)
    ax.bar(counts.index, counts.values, color=color, edgecolor='black')
    for i, v in enumerate(counts.values):
        ax.text(i, v + max(counts.values)*0.02, f'{int(v)}', ha='center', fontsize=10)
    ax.set_title(attr)
    ax.set_ylabel('Number of keyframes')
    pct_minor = 100 * min(counts.values) / max(counts.values, default=1)
    ax.set_xlabel(f'Imbalance ratio: {pct_minor:.0f}%')

plt.suptitle(f'Class distribution per attribute — {DATASET_VERSION}', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'class_distributions.png', bbox_inches='tight')
plt.show()

## 7. Cross-Attribute Co-Occurrence

If two attributes are highly correlated (e.g., night ↔ rain), we should know — it affects whether models can learn them independently.

In [ ]:
# Pairwise contingency tables and Cramér's V
from itertools import combinations

def cramers_v(x, y):
    """Cramér's V — categorical association strength, range [0, 1]."""
    confusion = pd.crosstab(x, y)
    chi2 = ((confusion - (confusion.sum(axis=1).values[:, None] * confusion.sum(axis=0).values / confusion.values.sum())) ** 2 /
            (confusion.sum(axis=1).values[:, None] * confusion.sum(axis=0).values / confusion.values.sum())).sum().sum()
    n = confusion.values.sum()
    r, k = confusion.shape
    denom = n * (min(r, k) - 1)
    return float(np.sqrt(chi2 / denom)) if denom > 0 else 0.0

print('Pairwise Cramér\'s V (0 = independent, 1 = perfectly associated):')
for a, b in combinations(attributes, 2):
    v = cramers_v(df_labels[a], df_labels[b])
    print(f'  {a:18s} ↔ {b:18s}: {v:.3f}')

In [ ]:
# Visualise: time_of_day × weather contingency
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [('time_of_day', 'weather'),
         ('time_of_day', 'vru_present'),
         ('weather', 'vehicle_density')]
for ax, (a, b) in zip(axes, pairs):
    ct = pd.crosstab(df_labels[a], df_labels[b], normalize='index')
    sns.heatmap(ct, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
                ax=ax, cbar=False)
    ax.set_title(f'{a} × {b}\n(row-normalised)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'attribute_cooccurrence.png', bbox_inches='tight')
plt.show()

## 8. Per-Scene Class Coverage

For scene-aware splitting later, we want every class of every attribute to appear in at least 2 scenes (otherwise we can't put one in train and one in test).

In [ ]:
print(f'Number of unique scenes: {df_labels["scene_token"].nunique()}')
print(f'\nClasses by number of scenes containing them:')
for attr in attributes:
    scene_class_pairs = df_labels[['scene_token', attr]].drop_duplicates()
    counts = scene_class_pairs[attr].value_counts().reindex(class_orders[attr]).fillna(0).astype(int)
    print(f'  {attr}:')
    for cls, n_scenes in counts.items():
        flag = '⚠ THIN' if n_scenes < 2 else 'ok'
        print(f'    {cls:10s}: {n_scenes:2d} scenes  [{flag}]')

---
## Findings & Decisions (fill in after running)

**Class distributions on v1.0-mini**
- Day vs night: _X_ vs _Y_ keyframes (imbalance ratio _Z_%)
- Clear vs rain: _X_ vs _Y_
- Vehicle density tertiles: edges at counts _A_ / _B_; resulting class sizes ~_C_ / _D_ / _E_
- VRU present: _X_ keyframes contain a VRU, _Y_ do not

**Cross-attribute associations**
- The strongest pairwise association is _attr_a_ ↔ _attr_b_ with Cramér's V = _value_.
- This [is / is not] high enough to be problematic — discuss in report.

**Scene-coverage flags**
- _N_ classes appear in fewer than 2 scenes; for those, scene-aware splitting becomes degenerate. Flag for notebook 04.

**Implications for next phase**
- Notebook 03 will extract image features for all _N_ keyframes.
- Notebook 04 will perform scene-aware train/val/test splitting and confirm class balance is preserved.
- Notebook 05 will train models on each attribute *independently* — the cross-attribute associations are interesting for analysis but not used at training time.
